# 03 — rSLDS on the IBL visual→motor triplet: is the decision a *sequence of attractors*?

**Regions: `VISp` → `SCm` → `MRN`**, same session as `01_hmm_triplet.ipynb` (`5c0c560e`), so the
two notebooks are directly comparable.

## What this notebook adds over the HMM

The Poisson HMM in notebook 01 tells us **which discrete state** the population is in at each bin,
and that motor areas are more *directed* than visual (SCm 0.77, MRN 0.69 vs VISp 0.53). It says
**nothing about what kind of dynamical object each state is.** An HMM state is just a firing-rate
pattern with a dwell time. It cannot distinguish:

- a genuine **point attractor** the population falls into and rests in,
- a **saddle** it merely passes through on its way somewhere else (a *heteroclinic waypoint* — the
  "winnerless competition" picture in `docs/project_notes/`),
- a **rotation**, or a **line attractor / integrator** — the shape a decision variable is supposed
  to live on.

An **rSLDS** (recurrent switching linear dynamical system) gives each discrete regime its own
**linear flow** on a continuous latent, so those objects become measurable: the flow's *fixed
point* and its *eigenvalues*. That is the object-level version of your claim.

$$z_t \mid z_{t-1}, x_{t-1} \sim \mathrm{softmax}\big(\log\Pi_{z_{t-1},:} + R\,x_{t-1}\big),
\qquad
x_t \mid z_t, x_{t-1} \sim \mathcal N\big(A_{z_t} x_{t-1} + b_{z_t},\; Q_{z_t}\big)$$

The $R\,x_{t-1}$ term is the **recurrence**, and it is the scientifically interesting part: it means
the population switches regime **because of where it is in state space**, not because a clock said
so. A model with $R = 0$ is "the brain runs a fixed timed script"; a model with $R \neq 0$ is "the
trajectory switches when it *arrives* somewhere". **We fit both and compare them on held-out
trials** — recurrence has to earn its parameters.

## The question, stated so it can fail

> *Sequence, not direction.* Do the regimes occur in a **reliable order** across trials?

Note this is deliberately **not** "does the sequence run forward". `forward_fraction` in notebook 01
already measures directedness *given* an imposed state ordering. Here we ask the weaker, more
honest question: when two regimes both occur, does one reproducibly come first — **whichever way
round**? That is `order_consistency` (`src/dynamics/rslds.py`):

- **0.5** → the population wanders among a *set* of states in no particular order. **No sequence.**
- **1.0** → every trial visits them in the same order. **A sequence.**

and it is tested against a **within-trial bin-shuffle null**, which preserves how much time each
trial spends in each regime and destroys only the order.

## Two backends, and why

| | `src/dynamics/rslds.py` (exact EM) | `src/dynamics/rslds_ssm.py` → Linderman `ssm` |
|---|---|---|
| latent | PCA on Anscombe counts, **fixed** | **inferred jointly** with the dynamics |
| emissions | Gaussian (on the latent) | **Poisson** (on the counts) |
| inference | exact forward-backward + weighted least squares | Laplace-EM (ELBO, a *lower bound*) |
| speed | seconds | **~15–20 min per region** |
| use here | model selection (K), recurrence test, screening | the **final, publishable fit** |

`ssm` is the published, tested method and it is the one whose numbers we report. But it is far too
slow to cross-validate K (≈13 h for the full grid), and — being a Laplace *lower bound* — its ELBO
is not an exact likelihood you can safely argmax over anyway. So the fast exact-EM model does model
selection and the recurrence test, and `ssm` is fitted at the chosen K. **If the two agree, the claim
is much stronger than either alone. If they disagree, believe `ssm`** and work out why the PCA latent
failed.

**No model selection is run here.** K is taken **per region** from the HMM state-stability analysis
(**VISp 3, SCm 4, MRN 3** — `docs/knowledge/best_K_per_region.md`), which is a *better* criterion
than held-out likelihood on this data: likelihood argmaxed to K=2 off a column of `-inf`, then to
K=10 off the top of the range. Part 4 explains this, and Part 9 (cross-validating K) is **off by
default** for exactly that reason.

> `ssm` lives in a **separate conda env** (`ndrh_ssm`): it needs numpy 1.x, and forcing that into
> `neurodatarehack` would drag pynapple/numba/pandas 3.0 down with it. Build it once with
> `bash scripts/install_ssm_env.sh` (**no admin rights needed** — it uses conda's mingw toolchain,
> which is why that script exists). The notebook calls it out-of-process and gets the same
> `RSLDSResult` back, so every figure below is backend-agnostic.

## Part 0 — Setup

Two timebases are built on purpose. **This is not fussiness — it is a result from ground truth.**
Validating `rslds.py` on synthetic data with *known* attractors (see the table in the module
docstring) showed that at **20 ms** bins with IBL-like sparse firing, the latent is so noisy that
the fitted $|\lambda|$ collapses onto $\approx 1$ and **every** regime gets labelled "line
attractor / integrator" — a near-unit-root random walk fitted to noise, not a finding. At **50 ms**
the same pipeline recovers genuinely contracting attractors ($|\lambda| = 0.875$, regime recovery
0.90). So:

- **20 ms** — matches notebook 01, keeps the HMM comparison honest.
- **50 ms** — the timebase the attractor read-out is actually trustworthy on.

We fit both and report both. **Any integrator/line-attractor claim from the 20 ms fit is not
admissible.**

In [ ]:
import sys, time, warnings, subprocess
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# ---------------------------------------------------------------------------------------
# KERNEL GUARD. This notebook must run in `neurodatarehack`.
#
# `ndrh_ssm` is the ISOLATED env that holds Linderman's ssm. It is numpy-1.x and has no
# ONE-api, no pynapple, no iblatlas -- it exists only so `ssm` can be imported by a
# subprocess. If VSCode picks it as the kernel (it will happily offer it), the imports below
# fail with a confusing ImportError instead of the real message, which is "wrong kernel".
# Fix: Ctrl+Shift+P -> "Notebook: Select Kernel" -> neurodatarehack.
# Do NOT `conda install -n ndrh_ssm ipykernel` to make this error go away -- that only makes
# it easier to run the notebook in the one env where it cannot work.
# ---------------------------------------------------------------------------------------
_env = Path(sys.prefix).name
if _env != "neurodatarehack":
    raise RuntimeError(
        f"Wrong kernel: this notebook is running in `{_env}`, but it needs `neurodatarehack`.\n"
        "  Ctrl+Shift+P -> 'Notebook: Select Kernel' -> neurodatarehack\n"
        "  (`ndrh_ssm` holds ssm only; the notebook calls it out-of-process, never runs in it.)"
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from src.data.access import get_one, sessions_table, load_trials, load_units
from src.data.build_matrices import (
    region_tensor_from_probes, region_spike_trains, drop_silent_units,
)
from src.dynamics.rslds import (
    counts_to_latents, fit_rslds, score_rslds, cross_validate_K,
    order_consistency, order_consistency_null, precedence_matrix,
    entry_times, visit_sequences, flow_field,
)
from src.dynamics.rslds_ssm import fit_rslds_ssm, ssm_env_available
from src.selection.msr import msr_table, rate_matched_control
from src.viz.style import (
    set_style, state_colors, region_color, savefig, stroke, SEQUENTIAL, DIVERGING,
)

set_style()

PARAMS   = yaml.safe_load((ROOT / "config" / "params.yaml").read_text())
CACHE    = ROOT / PARAMS["data"]["one_cache_dir"]
SESSION  = "5c0c560e-9e1f-45e9-b66e-e4ee7855be84"
REGIONS  = ["VISp", "SCm", "MRN"]        # frozen 2026-07-12; VISpm dropped
SEED     = PARAMS["random_seed"]
rng      = np.random.default_rng(SEED)

PRE_S, POST_S = PARAMS["binning"]["pre_s"], PARAMS["binning"]["post_s"]

# The two timebases (see the markdown above -- this choice is evidence-based, not taste).
BINS = {"20ms": 0.02, "50ms": 0.05}

# rSLDS knobs
N_LATENT  = 3      # chosen from the PCA variance curve in Part 3 -- NEVER by held-out LL (see there)
STICKY    = 2.0    # self-transition pseudo-counts; without this the regimes flicker (validated)
MAX_TRIALS = None  # None = all trials. Set e.g. 300 for a fast pass.

# K PER REGION, taken from the HMM state-stability analysis (chance-corrected ARI between two
# HMMs fitted to disjoint halves of the trials). NOT invented here, and NOT a likelihood argmax.
#   docs/knowledge/best_K_per_region.md  |  method: docs/knowledge/ARI_and_state_stability.md
# This is why Part 9 (cross-validation of K) is OFF by default: K is already chosen, by a better
# criterion than the one Part 9 would use.
K_PER_REGION = {
    "VISp": 3,   # ARI 0.883 +/- 0.019 (FINAL 2026-07-14; the old K=2 was a 1-restart artefact)
    "SCm":  4,   # ARI 0.928 +/- 0.003 -- the most reproducible structure in the dataset
    "MRN":  3,   # ARI 0.882 +/- 0.052
}

# Optional, slow, and NOT needed to look at the science: see Part 9 at the very bottom.
# Cost is len(CV_KS) x folds x restarts x 2 models fits PER region PER timebase, so it grows fast.
# Default to the 50 ms timebase only -- it is the one whose attractor read-out is admissible
# anyway (the 20 ms fit cannot be trusted for |lambda|), and it halves the wait.
RUN_CV    = False
CV_KS     = [2, 3, 4, 5, 6]
CV_BINS   = ["50ms"]      # set to list(BINS) to cross-validate both timebases (~2x slower)

print(f"ssm env (`ndrh_ssm`) available: {ssm_env_available()}")
print(f"regions: {REGIONS} | session: {SESSION[:8]} | timebases: {list(BINS)}")

## Part 1 — Load the session and build the count tensors

In [ ]:
t0 = time.time()
one  = get_one(cache_dir=CACHE)
pids = sessions_table(one).query("eid == @SESSION")["pid"].tolist()

trials, mask = load_trials(one, SESSION)
good = trials[mask].reset_index(drop=True)
if MAX_TRIALS is not None and len(good) > MAX_TRIALS:
    good = good.iloc[np.sort(rng.choice(len(good), MAX_TRIALS, replace=False))].reset_index(drop=True)

events = good["stimOn_times"].values
rt     = (good["firstMovement_times"] - good["stimOn_times"]).values
print(f"{len(good)} trials | median RT = {np.nanmedian(rt)*1000:.0f} ms   [{time.time()-t0:.1f}s]")

In [ ]:
# Good units on both probes, tagged with their Beryl region.
t0 = time.time()
from iblatlas.regions import BrainRegions
br = BrainRegions()

probe_loads = []
for pid in pids:
    spikes, clusters = load_units(one, pid)
    clusters = clusters.copy()
    clusters["Beryl"] = br.id2acronym(clusters["atlas_id"].values, mapping="Beryl")
    probe_loads.append({"spikes": spikes, "clusters": clusters})
print(f"{sum(p['clusters'].shape[0] for p in probe_loads)} good units   [{time.time()-t0:.1f}s]")

In [ ]:
# tensors[bin_label][region] -> (counts, bin_centers).  Raw integer counts: the Poisson rSLDS
# (like the Poisson HMM) needs them unsmoothed. Smoothing happens only inside counts_to_latents.
#
# `kept[label][region]` is NOT bookkeeping pedantry. drop_silent_units REMOVES columns and
# RE-INDEXES the rest 0..n_kept-1, while MSR (Part 2) scores units by their ORIGINAL id from
# region_spike_trains. Without this map the two indexings silently diverge and every
# "MSR-selected unit" would actually be a different neuron. Keep the map; use it.
tensors, kept = {}, {}
for label, bs in BINS.items():
    tensors[label], kept[label] = {}, {}
    for region in REGIONS:
        counts, bc = region_tensor_from_probes(
            probe_loads, region, events, pre_s=PRE_S, post_s=POST_S, bin_size_s=bs)
        n_before = counts.shape[2]
        counts, keep_idx = drop_silent_units(counts, min_total_spikes=1)
        tensors[label][region] = (counts, bc)
        kept[label][region] = keep_idx          # original unit id of each tensor column
    shapes = {r: tensors[label][r][0].shape for r in REGIONS}
    print(f"{label}: " + "  ".join(f"{r} {s}" for r, s in shapes.items()))
print("\n(trials, bins, units)")

## Part 2 — Multiscale Relevance: what `02_dimred_msr.ipynb` already settled

You asked for MSR as a unit-inclusion criterion. `src/selection/msr.py` exists and
**`02_dimred_msr.ipynb` has already validated and stress-tested it** — do not re-litigate that
here. What matters for *this* notebook is its three conclusions, because they constrain what we are
allowed to do with unit selection in an rSLDS:

1. **MSR is largely a firing-rate ranking on this data.** Spearman ρ(MSR, rate) = 0.64 (VISp),
   0.55 (SCm), 0.08 (MRN). The top-MSR half *is* mostly the loud half.
2. **The rate-matched control fails in VISp and SCm.** Once you take the loud half, the leftover
   pool has nothing loud enough to match it, so `rate_matched_control` silently degenerates into
   "the quiet half" (rate ratios 3.5× and 4.6×). **Only MRN has a trustworthy control.** Any
   MSR-vs-control comparison in VISp/SCm is a firing-rate difference in costume.
3. **In MRN — the one region where the test is fair — the top-MSR half was indistinguishable from a
   *random* half.** MSR bought nothing.

and, most directly relevant here:

> **"Choice information is redundant across the population; the temporal trajectory is not — it
> needs the whole population. Do not fit the HMM on half the units."** (worklog 2026-07-13)

That finding was made on *decoding*. **Part 8 below tests whether it also holds for the rSLDS
sequence statistic** — which is the thing we actually care about, and which nobody has checked yet.

### So: does MSR help us afford a smaller population?

**No — and Part 3 explains why that question is based on a false premise.** An rSLDS is limited by
**trials**, not by neurons. Pruning neurons buys no statistical power for the attractor structure
and *costs* latent signal-to-noise. MSR's honest job here is a **robustness axis**, not a budget.

In [ ]:
# Score the units once, for the Part 8 robustness split. `region_spike_trains` keys each unit by
# its COLUMN in region_tensor_from_probes -- but we then called drop_silent_units, which REMOVES
# columns and re-indexes the rest. `kept` is that map. Without it, "top-MSR unit 7" would point at
# a different neuron than tensor column 7, and nothing would look broken.
t0 = time.time()
trains = {r: region_spike_trains(probe_loads, r) for r in REGIONS}

MSR_BIN = "50ms"                                   # the tensor whose columns MSR is aligned to
msr_tables = {}
for region in REGIONS:
    tab = msr_table(trains[region])
    pos = {int(u): i for i, u in enumerate(kept[MSR_BIN][region])}
    tab["col"] = tab["unit"].map(pos)              # -> column in the post-drop tensor
    n_all = len(tab)
    tab = tab.dropna(subset=["col"]).astype({"col": int}).reset_index(drop=True)
    rho = tab[["msr", "rate_hz"]].corr(method="spearman").iloc[0, 1]
    msr_tables[region] = tab
    print(f"{region}: {len(tab)}/{n_all} units | MSR {tab.msr.min():.3f}-{tab.msr.max():.3f} | "
          f"rho(MSR, rate) = {rho:+.2f}")
print(f"[{time.time()-t0:.1f}s]")
print("\n(rho is the rate confound, reproduced from notebook 02. High rho => 'top MSR' == 'loud'.)")

## Part 3 — The latent space, and how many dimensions (D)

### D must NOT be chosen by held-out likelihood

This is the same class of trap as the `-inf` argmax that manufactured our bogus "K = 2 everywhere"
HMM result (2026-07-12): **a number that is not comparable being compared anyway.** Changing $D$
changes the *data* the Gaussian likelihood is computed on (a 2-D latent and a 3-D latent are
different observations), so their log-likelihoods are not on the same scale and the argmax is
meaningless. K may be cross-validated; **D may not.** We pick D from the variance-explained curve
and hold it fixed everywhere.

(For the `ssm` backend the emission is Poisson on the counts, so D *would* be comparable in
principle — but its ELBO is a lower bound whose tightness itself varies with D, so the argmax is
still not safe. Same conclusion.)

### The answer to your neurons-vs-trials question, empirically

Look at the variance explained below. That number *is* the neuron-count answer: it says how well
this many neurons pin down a low-dimensional latent per time bin.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
var_tab = []
for label, bs in BINS.items():
    for region in REGIONS:
        counts, bc = tensors[label][region]
        _, _, ve = counts_to_latents(counts, n_latent=8, smooth_sigma_bins=1.5 if bs == 0.02 else 1.0)
        a = ax[0] if label == "20ms" else ax[1]
        a.plot(np.arange(1, len(ve) + 1), np.cumsum(ve), marker="o", ms=4,
               color=region_color(region), label=region, path_effects=stroke())
        var_tab.append({"bin": label, "region": region, "n_units": counts.shape[2],
                        "var_exp_D3": round(float(np.sum(ve[:3])), 3),
                        "var_exp_D5": round(float(np.sum(ve[:5])), 3)})
for a, label in zip(ax, BINS):
    a.axvline(N_LATENT, color="0.4", ls="--", lw=1)
    a.set(xlabel="latent dimension D", ylabel="cumulative variance explained",
          title=f"PCA on Anscombe counts — {label}", ylim=(0, 1))
    a.legend(frameon=False, fontsize=9)
fig.tight_layout()
savefig(fig, "phase3_rslds", "pca_variance", root=ROOT)
plt.show()

display(pd.DataFrame(var_tab))
print(f"\nD = {N_LATENT} (dashed). Low cumulative variance is NOT necessarily a failure: at these")
print("firing rates most of the per-bin variance IS Poisson shot noise, and no latent should")
print("explain it. But it is exactly why the ssm backend (which models the counts as Poisson")
print("instead of pretending the noise is Gaussian) is the one we trust for the final numbers.")

### Does the rSLDS need many **neurons** or many **trials**? — the answer, and why it matters for MSR

Count the parameters, and notice *what each kind of data buys*:

| block | parameters | what estimates it |
|---|---|---|
| emissions $C, d$ | $N\!\times\!D + N$ | every time bin of **that neuron** |
| dynamics $A_k, b_k, Q_k$ | $K(D^2 + D + D^2)$ | all bins assigned to regime $k$ |
| transitions $\Pi, R$ | $K^2 + K D$ | **the switches actually observed** |

With $D=3, K=4$: dynamics + transitions ≈ **100 parameters**, and they are the scientific payload
(the attractors, and the sequence). They are estimated from the number of **regime transitions**,
and a trial contributes only a handful. **Trials are the currency of the switching dynamics.**

Neurons enter almost entirely through $C$ — and each extra neuron brings $D$ new parameters *plus a
whole new time series to constrain them*. So neurons are close to **free information**: they do not
compete for the data that estimates the attractors, they *improve* it, by sharpening the per-bin
estimate of $x_t$. That is what the variance-explained curve above is measuring.

**Concretely, for us:**

- **Trials are the binding constraint on the science.** ~800 trials × 20–50 bins ≈ 16k–40k bins,
  but only ~2–4 *switches* per trial → a few thousand transitions to fit $\Pi$ and $R$. That is
  adequate for $K \le 5$ and no more. If you want a stronger claim, **pool the 5 replicate sessions**
  (a hierarchical fit), do not add neurons.
- **Neurons set the single-trial SNR of $x_t$**, which is exactly what failed at 20 ms in the
  synthetic test. More neurons ⇒ a cleaner latent ⇒ $|\lambda|$ you can believe.

**Therefore: do not use MSR to prune neurons in the hope of buying statistical power. It buys none
— it can only cost you latent SNR.** Pruning is justified for (a) compute, (b) removing units whose
richness is artefactual (drift, bursty noise) and would distort a Gaussian latent, and (c) as the
**robustness check** we run below. That is a genuinely different reason from the one in the original
plan, where MSR was framed as a way to shrink the population for data-hungry methods (true for ELA,
**not** true for rSLDS).

## Part 4 — K comes from the HMM, not from this notebook

**No model selection is run here.** K is taken **per region** from the HMM state-stability
analysis — `docs/knowledge/best_K_per_region.md`:

| region | **K** | ARI (stability) | what it means |
|---|---|---|---|
| VISp | **2** | 0.975 ± 0.003 | a *negative* result — see below |
| SCm | **4** | **0.928 ± 0.003** | the most reproducible structure in the dataset |
| MRN | **3** | 0.870 ± 0.044 | real 3-state structure |

### Why this is a *better* criterion than the one Part 9 would use

Held-out likelihood has failed on this data twice, in opposite directions: it argmaxed to **K = 2**
off a column of `-inf`, then to **K = 10** off the top of the search range. Both are boundaries, not
answers. **Stability** asks a different and better question: fit two HMMs to *disjoint halves of the
trials* and measure how much they agree on the partition of time, chance-corrected (adjusted Rand
index). A state structure that does not reproduce across independent fits is not a state structure.

So Part 9 (cross-validating K) stays **off**. It would be a *worse* criterion applied to the same
question. It is still there for the **recurrence test**, which is a genuinely different question.

### Two things to carry into the fits below

1. **SCm K = 4 is the one to build on.** Two HMMs that never shared a trial agree on a four-way
   partition of time to within three thousandths.
2. **Do not expect a rich sequence in VISp.** Its ARI beyond the coarse baseline/evoked split is
   0.680 ± **0.176** — unstable. **VISp has ~2 reproducible states and no richer sequence**, and
   that is a finding, not a failure. **If the rSLDS reports many crisp, well-separated regimes in
   VISp, be suspicious** — an independent, chance-corrected criterion says they do not replicate.
   (Same class of error as *"20 ms bins manufacture line attractors"*: a model producing structure
   the data do not support.)

### Three caveats I am not going to paper over

- **These K are PRELIMINARY** (`n_repeats=2`, `n_restarts=1`) and, per that document's own §5,
  are **lower bounds** — `n_restarts=1` charges EM's bad luck to "the states are not real". The
  rigorous run may push SCm/MRN up.
- **They were measured on a different model at a different timebase**: a Poisson HMM at **20 ms**
  on rate-filtered units (`MIN_RATE_HZ = 1.0`). We are applying them to an **rSLDS at 50 ms** on
  *all* units. Discrete HMM states and rSLDS dynamical regimes are not the same object, and there
  is no guarantee the counts transfer. This is a **principled prior, not a measurement** — worth
  saying out loud.
- **VISp at K = 2 gives `order_consistency` exactly one regime pair to work with.** With a single
  pair the statistic is far noisier than at K = 3–4, and it cannot express a "sequence" richer than
  "A before B". Read VISp's row with that in mind.

In [ ]:
# K per region, from the HMM state-stability analysis (docs/knowledge/best_K_per_region.md).
# Same K at both timebases: the stability analysis was run at 20 ms, and we have no separate
# estimate for 50 ms -- so this is a transfer, and it is flagged as such in the markdown above.
BEST_K = {(label, region): K_PER_REGION[region] for label in BINS for region in REGIONS}

print("K per region (from HMM state stability, NOT selected here):")
for region in REGIONS:
    print(f"  {region:5s} K = {K_PER_REGION[region]}")
print("\nPart 9 (cross-validation of K) is OFF by default -- stability is the better criterion.")

## Part 5 — Fit both backends at the selected K

`ssm` is slow (~15–20 min per region on the full trial set). It is the fit we *report*; the exact-EM
fit is the fast cross-check. Set `RUN_SSM = False` to skip it while iterating.

In [ ]:
RUN_SSM   = True     # the slow, real one. ~15-20 min per region -- expect the better part of an hour.
SSM_ITERS = 50
SSM_BIN   = "50ms"   # the timebase the synthetic validation says the attractor read-out survives on

fits = {}
t0 = time.time()
for label in BINS:
    for region in REGIONS:
        counts, bc = tensors[label][region]
        smooth = 1.5 if BINS[label] == 0.02 else 1.0
        x, pca, ve = counts_to_latents(counts, n_latent=N_LATENT, smooth_sigma_bins=smooth)
        res = fit_rslds(x, BEST_K[(label, region)], recurrent=True, n_iter=60,
                        n_restarts=3, seed=SEED, sticky=STICKY)
        res.bin_centers = bc
        res.backend = "exact"
        fits[("exact", label, region)] = res
        sw = float(np.mean((np.diff(res.states, axis=1) != 0).sum(axis=1)))
        print(f"exact {label} {region:5s}: K={res.n_states}  switches/trial={sw:.1f}  "
              f"|lambda|max={np.abs(res.eigenvalues()).max():.3f}")
print(f"[exact backend: {time.time()-t0:.0f}s]")

In [ ]:
if RUN_SSM and ssm_env_available():
    t0 = time.time()
    for region in REGIONS:
        counts, bc = tensors[SSM_BIN][region]
        tr = time.time()
        res = fit_rslds_ssm(counts, n_states=BEST_K[(SSM_BIN, region)], n_latent=N_LATENT,
                            n_iter=SSM_ITERS, seed=SEED, root=ROOT, verbose=True)
        res.bin_centers = bc
        fits[("ssm", SSM_BIN, region)] = res
        sw = float(np.mean((np.diff(res.states, axis=1) != 0).sum(axis=1)))
        print(f"ssm {SSM_BIN} {region:5s}: K={res.n_states}  switches/trial={sw:.1f}  "
              f"|lambda|max={np.abs(res.eigenvalues()).max():.3f}  [{time.time()-tr:.0f}s]\n")
    print(f"[ssm backend total: {time.time()-t0:.0f}s]")
elif RUN_SSM:
    print("`ndrh_ssm` env not found -- build it with `bash scripts/install_ssm_env.sh`.")
    print("Continuing with the exact-EM backend only.")

## Part 6 — What kind of dynamical object is each regime?

For each regime $k$ the flow is $x_{t+1} = A_k x_t + b_k$, so:

- **fixed point** $x^*_k = (I - A_k)^{-1} b_k$ — where the flow would come to rest;
- **eigenvalues** of $A_k$ — $|\lambda| < 1$ contracting (**attractor**), $|\lambda| > 1$ expanding,
  a mix ⇒ **saddle** (a waypoint the trajectory is pushed *through* — the heteroclinic picture),
  complex ⇒ rotation, $|\lambda| \approx 1$ ⇒ a **slow / integrator** direction;
- **timescale** $\tau = -\Delta t / \ln|\lambda_{\max}|$ — a $\tau$ of a few bins is a transient; a
  $\tau$ longer than the trial is effectively an integrator.

> **Read this with the 20 ms caveat in hand.** Ground truth said that at 20 ms the low-SNR latent
> drives $|\lambda| \to 1$ and *manufactures* "line attractor" labels. If the 20 ms and 50 ms rows
> disagree in exactly that direction, the 20 ms row is the artefact.

In [ ]:
rows = []
for (backend, label, region), res in fits.items():
    bs = BINS[label]
    fp, ev, tau, lab = res.fixed_points(), res.eigenvalues(), res.timescales(bs), res.classify()
    occ = res.gamma.mean(axis=(0, 1))
    for k in range(res.n_states):
        rows.append({
            "backend": backend, "bin": label, "region": region, "regime": k,
            "occupancy": round(float(occ[k]), 3),
            "|lambda|max": round(float(np.abs(ev[k]).max()), 3),
            "tau_ms": None if not np.isfinite(tau[k]) else round(float(tau[k] * 1000)),
            "rotational": bool(np.any(np.abs(ev[k].imag) > 1e-6)),
            "kind": lab[k],
        })
attr = pd.DataFrame(rows)
display(attr)
(ROOT / "results" / "rslds").mkdir(parents=True, exist_ok=True)
attr.to_csv(ROOT / "results" / "rslds" / "attractor_classification.csv", index=False)
print("saved -> results/rslds/attractor_classification.csv")

In [ ]:
# Flow fields: the money figure. Latent trajectories coloured by regime, with each regime's
# vector field and fixed point drawn on top.
def plot_flows(backend, label):
    keys = [(backend, label, r) for r in REGIONS if (backend, label, r) in fits]
    if not keys:
        return None
    fig, ax = plt.subplots(1, len(keys), figsize=(5.4 * len(keys), 5))
    for a, key in zip(np.atleast_1d(ax), keys):
        res = fits[key]
        region, K = key[2], fits[key].n_states
        cols = state_colors(K)
        pts = res.x.reshape(-1, res.x.shape[2])
        zz  = res.states.ravel()

        for k in range(K):
            m = zz == k
            a.scatter(pts[m, 0], pts[m, 1], s=2, color=cols[k], alpha=0.25, zorder=1)
            GX, GY, U, V = flow_field(res, k, n_grid=13)
            a.quiver(GX, GY, U, V, color=cols[k], alpha=0.55, width=0.003, zorder=2)

        fps = res.fixed_points()
        for k in range(K):
            if not np.all(np.isfinite(fps[k])):
                continue
            inside = (pts[:, :2].min(0) <= fps[k, :2]).all() and (fps[k, :2] <= pts[:, :2].max(0)).all()
            a.scatter(*fps[k, :2], s=190, marker="*", color=cols[k], edgecolor="black",
                      linewidth=1.2, zorder=5,
                      label=f"regime {k}: {res.classify()[k]}" + ("" if inside else "  (outside data)"))
        a.set(xlabel="latent 1", ylabel="latent 2", title=f"{region} — {backend}, {label}")
        a.legend(frameon=False, fontsize=7.5, loc="best")
    fig.tight_layout()
    return fig

for backend, label in [("exact", "50ms"), ("ssm", SSM_BIN)]:
    f = plot_flows(backend, label)
    if f is not None:
        savefig(f, "phase3_rslds", f"flow_fields_{backend}_{label}", root=ROOT)
        plt.show()
print("A fixed point far OUTSIDE the cloud of visited latent states is not an attractor the")
print("population ever reaches -- it is an extrapolation of a linear fit. Those are flagged.")

## Part 7 — The actual question: is there a **sequence** (direction-agnostic)?

`order_consistency` averages $\max\big(P[k,l],\,1-P[k,l]\big)$ over regime pairs, where $P[k,l]$ is
the fraction of trials in which regime $k$ is first entered before regime $l$. It asks *"is one
reliably first, whichever way round"*, and is **blind to which order** — so it tests
**sequence**, not direction, exactly as you framed it.

**The null is not 0.5.** Shuffling bins within a trial preserves each trial's regime composition
and destroys the order, but it lands at ≈0.53–0.62, not 0.5, because shuffling still changes *which*
regime happens to be entered first. So we compare to the **shuffle null**, never to 0.5 — and we
report the **effect size** alongside the p-value, because with hundreds of trials a trivial order
bias is still "significant".

In [ ]:
t0 = time.time()
seq_rows = []
for (backend, label, region), res in fits.items():
    K  = res.n_states
    oc = order_consistency(res.states, K)
    null = order_consistency_null(res.states, K, n_shuffles=200, seed=SEED)
    p = float((null >= oc).mean())
    z = (oc - null.mean()) / (null.std() + 1e-12)
    seq_rows.append({
        "backend": backend, "bin": label, "region": region, "K": K,
        "order_consistency": round(oc, 3),
        "null_mean": round(float(null.mean()), 3),
        "effect (OC - null)": round(oc - float(null.mean()), 3),
        "z": round(float(z), 1),
        "p": p,
        "switches/trial": round(float(np.mean((np.diff(res.states, axis=1) != 0).sum(axis=1))), 1),
    })
seq = pd.DataFrame(seq_rows).sort_values(["backend", "bin", "region"])
display(seq)
seq.to_csv(ROOT / "results" / "rslds" / "sequence_stats.csv", index=False)
print(f"saved -> results/rslds/sequence_stats.csv   [{time.time()-t0:.0f}s]")
print("\nInterpretation: a LARGE `effect` with a small p is a real sequence. A tiny `effect` with")
print("a small p is a statistically-detectable triviality -- do not sell it as a sequence.")

In [ ]:
# Precedence matrices: WHICH regime comes before which. Diverging map centred on 0.5 = no order.
for backend, label in [("exact", "50ms"), ("ssm", SSM_BIN)]:
    keys = [(backend, label, r) for r in REGIONS if (backend, label, r) in fits]
    if not keys:
        continue
    fig, ax = plt.subplots(1, len(keys), figsize=(4.3 * len(keys), 4))
    for a, key in zip(np.atleast_1d(ax), keys):
        res = fits[key]
        P = precedence_matrix(res.states, res.n_states)
        im = a.imshow(P, cmap=DIVERGING, vmin=0, vmax=1)
        for i in range(res.n_states):
            for j in range(res.n_states):
                if np.isfinite(P[i, j]):
                    a.text(j, i, f"{P[i, j]:.2f}", ha="center", va="center", fontsize=8)
        a.set(xlabel="regime l", ylabel="regime k", title=f"{key[2]} — P(k before l)",
              xticks=range(res.n_states), yticks=range(res.n_states))
        fig.colorbar(im, ax=a, fraction=0.046)
    fig.suptitle(f"precedence — {backend}, {label}   (0.5 = no reliable order)", y=1.03)
    fig.tight_layout()
    savefig(fig, "phase3_rslds", f"precedence_{backend}_{label}", root=ROOT)
    plt.show()

In [ ]:
# Regime entry times, and the CROSS-REGION ordering -- the inter-areal read-out.
for backend, label in [("exact", "50ms"), ("ssm", SSM_BIN)]:
    keys = [(backend, label, r) for r in REGIONS if (backend, label, r) in fits]
    if not keys:
        continue
    print(f"\n===== {backend}, {label} =====")
    for key in keys:
        res = fits[key]
        et = entry_times(res.states, res.bin_centers, res.n_states)
        et.insert(0, "region", key[2])
        display(et)

    # The most common regime path per region -- the sequence, written out.
    for key in keys:
        from collections import Counter
        seqs = visit_sequences(fits[key].states)
        top = Counter(tuple(s) for s in seqs).most_common(3)
        n = len(seqs)
        print(f"{key[2]:5s} most common regime paths: " +
              "   ".join(f"{list(s)} ({100*c/n:.0f}%)" for s, c in top))

In [ ]:
# Regime occupancy over time, per region -- the visual form of "a sequence of states".
for backend, label in [("exact", "50ms"), ("ssm", SSM_BIN)]:
    keys = [(backend, label, r) for r in REGIONS if (backend, label, r) in fits]
    if not keys:
        continue
    fig, ax = plt.subplots(1, len(keys), figsize=(5.2 * len(keys), 3.8), sharey=True)
    for a, key in zip(np.atleast_1d(ax), keys):
        res = fits[key]
        occ = res.gamma.mean(axis=0).T                   # (K, T)
        cols = state_colors(res.n_states)
        for k in range(res.n_states):
            a.plot(res.bin_centers, occ[k], color=cols[k], lw=1.8, label=f"regime {k}",
                   path_effects=stroke())
        a.axvspan(res.bin_centers[0], 0, color="0.92", zorder=0)
        a.axvline(0, color="0.3", ls="--", lw=1)
        a.axvline(np.nanmedian(rt), color="black", ls=":", lw=1.2)
        a.set(xlabel="time from stimOn (s)", title=key[2])
        a.legend(frameon=False, fontsize=8, ncol=2)
    np.atleast_1d(ax)[0].set_ylabel("P(regime)")
    fig.suptitle(f"regime occupancy — {backend}, {label}   "
                 f"(grey = pre-stimulus baseline; dotted = median RT)", y=1.04)
    fig.tight_layout()
    savefig(fig, "phase3_rslds", f"occupancy_{backend}_{label}", root=ROOT)
    plt.show()

## Part 8 — Does the **sequence** survive dropping half the neurons?

Notebook 02 showed that halving the population is nearly free for **decoding choice** but costs you
the **temporal trajectory** (MRN time-R² 0.550 → 0.483), and concluded: *"Selection is fine for
decoding, not for dynamics."* That was measured on decoding. **Here we test it on the thing this
project actually claims** — the sequence of attractor regimes.

Refit at 50 ms on three unit sets: **all**, **top-50% MSR**, and a **rate-matched** control.

> **Carry notebook 02's warning with you.** The rate-matched control **fails in VISp and SCm** —
> once the loud half is taken, the remaining pool has nothing loud enough to match it, so the
> "control" degenerates into the quiet half. The cell below therefore prints the achieved rate
> ratio and marks each control `matched` / `NOT matched`. **Only rows marked matched are evidence.**
> On this session that is essentially **MRN only**.

Prediction, if notebook 02's conclusion generalises: `order_consistency` should **drop** when
neurons are removed — because fewer neurons means a noisier latent, and a noisy latent blurs the
regime boundaries. That is the neurons-vs-trials story from Part 3, and this is where it gets
tested rather than asserted.

In [ ]:
LABEL_RB = MSR_BIN          # msr_tables' `col` is aligned to this tensor
rb_rows = []
for region in REGIONS:
    counts, bc = tensors[LABEL_RB][region]

    # Work in TENSOR-COLUMN space throughout: rate_matched_control selects from the same
    # `unit` column it is handed, and those ids are then used to slice `counts`.
    tab = (msr_tables[region].dropna(subset=["msr"])
           .drop(columns=["unit"]).rename(columns={"col": "unit"}))
    if len(tab) < 8:
        print(f"{region}: too few scoreable units for the robustness split — skipped")
        continue

    n_keep  = max(len(tab) // 2, 4)
    top_msr = tab.nlargest(n_keep, "msr")["unit"].to_numpy()
    matched = rate_matched_control(tab, top_msr, seed=SEED)

    # Did the "rate-matched" control actually match? (Notebook 02: in VISp/SCm it does not.)
    rate_of = tab.set_index("unit")["rate_hz"]
    r_top, r_ctl = float(rate_of.loc[top_msr].median()), float(rate_of.loc[matched].median())
    ratio = max(r_top, r_ctl) / max(min(r_top, r_ctl), 1e-9)
    is_matched = ratio < 1.5
    print(f"\n{region}: msr_top median {r_top:.2f} Hz vs control {r_ctl:.2f} Hz "
          f"(ratio {ratio:.1f}x) -> control {'MATCHED' if is_matched else 'NOT matched — not evidence'}")

    for tag, units in [("all", np.arange(counts.shape[2])),
                       ("top50%_MSR", top_msr),
                       ("rate_matched", matched)]:
        if len(units) < 4:
            continue
        x, _, ve = counts_to_latents(counts[:, :, units], n_latent=N_LATENT, smooth_sigma_bins=1.0)
        res = fit_rslds(x, BEST_K[(LABEL_RB, region)], recurrent=True, n_iter=60,
                        n_restarts=3, seed=SEED, sticky=STICKY)
        oc = order_consistency(res.states, res.n_states)
        null = order_consistency_null(res.states, res.n_states, n_shuffles=100, seed=SEED)
        rb_rows.append({
            "region": region, "unit_set": tag, "n_units": len(units),
            "var_exp": round(float(ve.sum()), 3),
            "order_consistency": round(oc, 3),
            "effect (OC - null)": round(oc - float(null.mean()), 3),
            "|lambda|max": round(float(np.abs(res.eigenvalues()).max()), 3),
            # only meaningful for the control row, but carried per-region for filtering
            "control_matched": bool(is_matched) if tag == "rate_matched" else None,
        })
        print(f"  {tag:14s} n={len(units):3d}  var_exp={ve.sum():.3f}  OC={oc:.3f}  "
              f"effect={oc - float(null.mean()):+.3f}")

rb = pd.DataFrame(rb_rows)
display(rb)
rb.to_csv(ROOT / "results" / "rslds" / "msr_robustness.csv", index=False)
print("saved -> results/rslds/msr_robustness.csv")
print("\nRead the `all` -> `top50%_MSR` change: if order_consistency FALLS, then removing neurons")
print("degraded the latent and blurred the regimes -- confirming, now on the DYNAMICS rather than")
print("on decoding, notebook 02's rule: select units for decoding, never for state-space models.")

## Part 9 (OPTIONAL, slow) — the **recurrence test**, and a likelihood cross-check on K

Set `RUN_CV = True` in Part 0. Two separate things come out of this cell, and they are **not** of
equal value:

**1. The recurrence test — this is the one worth running.**
`delta_recurrent` = (held-out LL with $R$ free) − (held-out LL with $R = 0$). $> 0$ means switching
genuinely depends on **where the latent is**: the population leaves a state because it *arrives*
somewhere, not because a clock fired. **That is the entire difference between an attractor sequence
and a stopwatch**, and nothing else in this notebook tests it. It is the single most interesting
number here.

**2. K by held-out likelihood — a cross-check only, and a weak one.**
We already have K from a **better** criterion (stability, Part 4). Held-out likelihood has failed
twice on this data — K=2 off a column of `-inf`, then K=10 off the top of the range. So treat any
disagreement with Part 4 as *evidence about the likelihood criterion*, not as a reason to change K.

The PCA basis is **re-fitted inside every fold on the training trials only**. Fitting it once on all
trials would leak the test fold into the latent space and inflate every held-out score.

> **Read the curve, not the argmax.** If held-out LL is flat across K, "best K" is noise, and the
> honest report is "the data do not prefer a K". If the argmax lands on the **edge** of `CV_KS`, the
> curve is still climbing and the optimum is outside the window — the cell says so rather than
> pretending it selected something.

### Why this is no longer a 45-minute cell

It was, and that was a **bug**, not a cost of the method: the transition M-step ran L-BFGS to full
convergence *inside every EM iteration*. Profiled on the real session (843 trials × 50 bins, K=4),
**96% of all runtime** sat in that one call — and the final log-likelihood was **bit-identical** on
a fraction of the budget:

| L-BFGS budget | fit time | final LL/bin |
|---|---|---|
| 100 (naive) | 40.8 s | −0.6362 |
| 20 | 18.0 s | −0.6362 |
| **10 (now)** | **~9 s** | **−0.6362** |

EM only requires the M-step to *increase* the objective, not to maximise it — the next E-step
re-optimises anyway (textbook **generalised EM**). Re-verified against the synthetic ground truth:
regime recovery, |λ| and order-consistency all unchanged. Measured: **45 min → ~7 min**.

In [ ]:
if not RUN_CV:
    print("RUN_CV = False -- skipped.")
    print(f"K came from HMM state stability: {K_PER_REGION} (docs/knowledge/best_K_per_region.md).")
    print("\nSet RUN_CV = True to run the RECURRENCE test (delta_recurrent) -- that one is not")
    print("optional science, it is the only test of 'attractor sequence vs stopwatch' we have.")
else:
    n_fits = len(CV_KS) * 3 * 2 * 2 * len(REGIONS) * len(CV_BINS)   # K x folds x restarts x models
    print(f"cross-validating K over {CV_KS} on {CV_BINS} -> ~{n_fits} fits. "
          f"Expect roughly {n_fits * 6 / 60:.0f} min.\n")
    t0 = time.time()
    cv = {}
    for label in CV_BINS:
        for region in REGIONS:
            counts, _ = tensors[label][region]
            smooth = 1.5 if BINS[label] == 0.02 else 1.0
            tr = time.time()
            df = cross_validate_K(counts, CV_KS, n_latent=N_LATENT, smooth_sigma_bins=smooth,
                                  n_folds=3, n_iter=40, n_restarts=2, seed=SEED,
                                  sticky=STICKY, compare_recurrent=True)
            cv[(label, region)] = df
            k_best = int(df.loc[df.mean_test_ll.idxmax(), "n_states"])
            d_rec  = float(df.loc[df.mean_test_ll.idxmax(), "delta_recurrent"])

            # Is the curve actually preferring a K, or is it flat? A "best K" whose advantage is
            # smaller than the fold-to-fold scatter is not a selection, it is noise.
            spread = float(df.mean_test_ll.max() - df.mean_test_ll.min())
            noise  = float(df.std_test_ll.mean())
            flat = spread < noise
            note = "  <- FLAT (spread < fold noise): argmax is NOISE, do not select on it" if flat else ""
            if k_best == max(CV_KS):
                note = "  <- argmax at the EDGE of CV_KS: curve still climbing, optimum is outside"
            print(f"{label} {region:5s}: best K = {k_best}  delta_recurrent = {d_rec:+.4f}  "
                  f"[{time.time()-tr:.0f}s]{note}")
    print(f"\n[total {time.time()-t0:.0f}s]")

In [ ]:
if RUN_CV:
    fig, ax = plt.subplots(2, len(CV_BINS), figsize=(6 * len(CV_BINS), 8), squeeze=False)
    for j, label in enumerate(CV_BINS):
        for region in REGIONS:
            df = cv[(label, region)]
            c = region_color(region)
            ax[0, j].errorbar(df.n_states, df.mean_test_ll, yerr=df.std_test_ll, marker="o", ms=5,
                              capsize=3, lw=1.8, color=c, label=region, path_effects=stroke())
            # each region used its OWN K (from stability) -- mark them separately
            ax[0, j].axvline(K_PER_REGION[region], color=c, ls="--", lw=1.2, alpha=0.7)
            ax[1, j].plot(df.n_states, df.delta_recurrent, marker="s", ms=5, lw=1.8, color=c,
                          label=region, path_effects=stroke())
        ax[0, j].set(xlabel="K (regimes)", ylabel="held-out LL / bin",
                     title=f"likelihood vs K — {label}\n(dashed = K used, from HMM stability)")
        ax[0, j].legend(frameon=False, fontsize=9)
        ax[1, j].axhline(0, color="0.3", ls="--", lw=1)
        ax[1, j].set(xlabel="K (regimes)", ylabel="LL(recurrent) - LL(R=0)",
                     title="THE RECURRENCE TEST:\nswitching depends on latent position?  (>0 = yes)")
        ax[1, j].legend(frameon=False, fontsize=9)
    fig.tight_layout()
    savefig(fig, "phase3_rslds", "cv_K_and_recurrence", root=ROOT)
    plt.show()

    print("RECURRENCE (the number that matters) at the K we actually used:")
    for label in CV_BINS:
        for region in REGIONS:
            df = cv[(label, region)]
            row = df[df.n_states == K_PER_REGION[region]]
            if len(row):
                d = float(row.delta_recurrent.iloc[0])
                verdict = "position-dependent switching" if d > 0 else "no better than a stopwatch"
                print(f"  {label} {region:5s} K={K_PER_REGION[region]}: "
                      f"delta_recurrent = {d:+.4f}  -> {verdict}")

    cv_k = {(l, r): int(cv[(l, r)].loc[cv[(l, r)].mean_test_ll.idxmax(), "n_states"])
            for l in CV_BINS for r in REGIONS}
    print("\nlikelihood-argmax K (cross-check only):", {f"{l}/{r}": k for (l, r), k in cv_k.items()})
    print(f"stability K (what we used):            {K_PER_REGION}")
    print("\nDisagreement here is NOT a reason to change K -- held-out likelihood has already")
    print("failed twice on this data (K=2 off -inf; K=10 off the top of the range). It is")
    print("evidence about the criterion. Stability compares INDEPENDENT fits; likelihood cannot.")

## Caveats — what this notebook does and does not license you to claim

1. **One session, one animal.** The triplet `{VISp, SCm, MRN}` is co-recorded in **5** IBL sessions.
   Nothing here is a result until it replicates across them. This notebook is the machinery.

2. **The 20 ms attractor labels are not admissible.** Ground truth (see the table in
   `src/dynamics/rslds.py`) showed that at 20 ms with IBL-like sparse firing the latent is noise-
   dominated, $|\lambda| \to 1$, and *every* regime is labelled "line attractor / integrator". If
   you see that pattern at 20 ms and not at 50 ms, it is the artefact, not a decision variable.

3. **`ssm`'s number is an ELBO, not a likelihood.** It is a *lower bound*. Do not compare ELBOs
   across different D or K and call it model selection — that is why K is chosen with the exact-EM
   backend, whose likelihood really is exact.

4. **The exact-EM backend's latent is PCA, and PCA optimises variance, not dynamics.** A
   low-variance direction can still be dynamically decisive, and PCA will throw it away. This is the
   one place where `ssm` is strictly better, and it is why `ssm` is the backend of record.

5. **`order_consistency` tests sequence, not causation, and not flow.** A reliable order within a
   region is consistent with a common input driving both regions on a schedule. The cross-area claim
   still needs the **joint multi-region model** (SCm+MRN on one concatenated population) — that is
   the next step, and the rSLDS machinery here is exactly what it needs.

6. **MSR is largely a firing-rate filter on this data** (notebook 02: ρ = 0.64 / 0.55 / 0.08, and in
   the one region with a fair control it did not beat *random* selection). Use it as a robustness
   axis. **Do not use it to "afford" fewer neurons** — an rSLDS is limited by **trials**, not
   neurons (Part 3), so pruning neurons buys no statistical power and costs latent SNR. Part 8
   tests that directly on the sequence statistic.

7. **The rate-matched control is broken in VISp and SCm** and cannot be fixed within one session —
   there are simply not enough loud units left over to match. Pooling the 5 replicate triplet
   sessions is the only real fix. Until then, MRN is the only region where an MSR-vs-control
   comparison means anything.

8. **K is imported, not measured here** (VISp 2, SCm 4, MRN 3, from HMM state stability). Two
   honest limits on that: those numbers are **preliminary** and are **lower bounds** by their own
   document's §5 (`n_restarts=1` charges EM's bad luck to "the states are not real"); and they were
   measured on a **different model at a different timebase** (Poisson HMM, 20 ms, rate-filtered
   units) than the one we apply them to (rSLDS, 50 ms, all units). It is a principled prior, not a
   measurement.

9. **The recurrence test has not been run** (Part 9, off by default). Until it is, we have **not**
   shown that switching depends on latent *position* rather than elapsed *time* — and that is the
   whole difference between an attractor sequence and a stopwatch. Everything in Parts 5–8 is
   compatible with a population that simply runs a timed script. **Run Part 9 before claiming
   "attractor".**

10. **VISp's K = 2 makes its `order_consistency` a single regime pair.** The statistic is much
    noisier there and cannot express anything richer than "A before B". And per Part 4, VISp has no
    reproducible structure beyond the coarse split — **if the rSLDS shows crisp multi-regime
    sequences in VISp, distrust them.**

## Next

- **The joint SCm+MRN rSLDS.** Fit one rSLDS on the *concatenated* SCm+MRN population and ask
  whether the regimes are shared, and whether the recurrence matrix $R$ carries cross-area
  structure. That is the direct test of your feedback hypothesis, and it is the one thing the
  per-region fits here cannot do.
- **Replicate across the 5 triplet sessions**; pool trials rather than neurons.
- Consider `sticky` as a *fitted* hyper-parameter rather than a fixed 2.0, chosen by held-out LL.

# Part 10 — Results (2026-07-15, single session, unified 40 ms pipeline)

**This section is self-contained** and does not depend on Parts 1–9 above (which predate the
2026-07-15 unification and still build their own tensors). Run it from a fresh kernel: it loads the
**frozen 40 ms tensors** (`data/processed/`, from `00_preprocessing.ipynb`) and the result tables
written by the analysis scripts, and does its own quick fits.

**What was tested, and the verdict** (full write-up: `docs/worklog/2026-07-15_rslds-first-real-test.md`):

| result | status | evidence |
|---|---|---|
| **Attractor *sequence*, not a stopwatch** | ✅ **CERTIFIED** | `delta_recurrent > 0` every region, both alignments; a clean **position-shuffle null** passes in 100 % of folds |
| **A reliable temporal sequence exists** | ✅ CERTIFIED | `order_consistency ≫` its shuffle null, both backends |
| **Visual→motor gradient** | ✅ robust | recurrence + position-dependence ordered **MRN ≈ SCm ≫ VISp** |
| **Line-attractor / integrator geometry** | ❌ **RETIRED** | `|λ|`-stability: `|λ|≈1` is **>97 % reproduced on temporally-shuffled data** → the *smoother*, not the dynamics |

> **Honest headline:** decision-making traverses a **position-dependent sequence of states**,
> strongest in the motor areas — but each attractor's *shape* is **not identifiable** from one
> session at these rates (every single-trial latent imposes smoothness that the eigenvalues read
> back as a slow mode). GPFA's **finite** τ (107–283 ms, motor slowest) is the honest read on "how
> slow". The plots below visualise the script outputs plus a live fit for the trajectories.


In [ ]:
# --- self-contained setup: frozen 40 ms tensors + the result JSONs ---
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data.preprocess import WORKING_EID, load_processed
from src.dynamics.rslds import counts_to_latents, fit_rslds
from src.viz.style import set_style, region_color, state_colors, savefig, stroke, OKABE_ITO
set_style()
PHASE, REGIONS = "phase3_rslds_results", ["VISp", "SCm", "MRN"]

RES = ROOT / "results" / "rslds"
def _load(name):
    p = RES / name
    return json.loads(p.read_text()) if p.exists() else None
screen  = _load("screen_40ms.json")
recur   = _load("recurrence_test.json")
lamstab = _load("lambda_stability.json")
ssmrec  = _load("ssm_record_40ms.json")
for nm, obj in [("screen", screen), ("recurrence", recur), ("lambda_stability", lamstab), ("ssm", ssmrec)]:
    print(f"  {nm:16s} {'loaded' if obj else 'MISSING - run the matching script'}")

# frozen 40 ms tensor, stimOn (the primary alignment)
D = load_processed(ROOT / "data" / "processed" / f"{WORKING_EID[:8]}__stimOn_pre400_post800_bin40_rate1.npz")
BC = D["bin_centers"]
K_PER_REGION = {"VISp": 3, "SCm": 4, "MRN": 3}
print(f"\nfrozen tensor: {D['n_trials']} trials, {len(BC)} bins ({BC[0]:+.3f}..{BC[-1]:+.3f} s), "
      f"units {[D['regions'][r]['n_units'] for r in REGIONS]}")


In [ ]:
# ============================================================================
# FIGURE 1 — the certified results (and the retired one), from the script JSONs
# ============================================================================
sdf = pd.DataFrame(screen["rows"]);  rdf = pd.DataFrame(recur["rows"])
aligns = ["stimOn", "firstMovement"]

fig, ax = plt.subplots(2, 2, figsize=(13, 9))
x = np.arange(len(REGIONS)); w = 0.38

# (a) recurrence: delta_recurrent, both alignments, per-fold SD error bars
for j, al in enumerate(aligns):
    d = rdf[rdf.alignment == al].set_index("region").loc[REGIONS]
    ax[0,0].bar(x + (j-0.5)*w, d.delta_mean, w, yerr=d.delta_sd, capsize=3,
                color=[region_color(r) for r in REGIONS], alpha=0.75 if j else 1.0,
                edgecolor="black", lw=0.5, hatch="" if j==0 else "//",
                label="stimOn" if j==0 else "firstMovement")
ax[0,0].axhline(0, color="0.2", lw=1)
ax[0,0].set(xticks=x, xticklabels=REGIONS, ylabel="held-out LL(R free) − LL(R=0)  [nats/bin]",
            title="(a) Recurrence: attractor SEQUENCE, not stopwatch\n(>0 = switching depends on latent position)")
ax[0,0].legend(frameon=False, fontsize=8)

# (b) position-shuffle: the clean null. real − shuffled-position, both alignments
for j, al in enumerate(aligns):
    d = rdf[rdf.alignment == al].set_index("region").loc[REGIONS]
    ax[0,1].bar(x + (j-0.5)*w, d.real_minus_posnull_mean, w,
                color=[region_color(r) for r in REGIONS], alpha=0.75 if j else 1.0,
                edgecolor="black", lw=0.5, hatch="" if j==0 else "//")
ax[0,1].axhline(0, color="0.2", lw=1)
ax[0,1].set(xticks=x, xticklabels=REGIONS,
            ylabel="real − position-shuffled  [nats/bin]",
            title="(b) Switching depends on WHERE the latent is\n(100% of folds: real > shuffled — the clean null)")
ax[0,1].text(0.5, 0.92, "gradient  MRN ≈ SCm ≫ VISp", transform=ax[0,1].transAxes,
             ha="center", fontsize=9, color="0.3")

# (c) order_consistency vs its shuffle null (stimOn)
s = sdf[sdf.alignment == "stimOn"].set_index("region").loc[REGIONS]
ax[1,0].bar(x, s.order_consistency, 0.6, color=[region_color(r) for r in REGIONS],
            edgecolor="black", lw=0.5, zorder=3)
for i, r in enumerate(REGIONS):
    ax[1,0].plot([i-0.35, i+0.35], [s.loc[r,"oc_null_p95"]]*2, color="0.2", lw=1.5, zorder=4)
    ax[1,0].add_patch(plt.Rectangle((i-0.35, s.loc[r,"oc_null_mean"]), 0.7,
                      s.loc[r,"oc_null_p95"]-s.loc[r,"oc_null_mean"], color="0.75", alpha=0.6, zorder=2))
ax[1,0].axhline(0.5, color="0.5", ls=":", lw=1)
ax[1,0].set(xticks=x, xticklabels=REGIONS, ylim=(0.5, 1.0), ylabel="order_consistency",
            title="(c) A reliable SEQUENCE exists\n(bar = observed; grey = shuffle null, line = null p95)")

# (d) |lambda|: real vs shuffle -> the geometry is the smoother (RETIRED)
lam = lamstab["summary"]
real = [lam[r]["real_mean"] for r in REGIONS]; shuf = [lam[r]["shuffle_mean"] for r in REGIONS]
ax[1,1].bar(x - w/2, real, w, label="real data", color=[region_color(r) for r in REGIONS],
            edgecolor="black", lw=0.5)
ax[1,1].bar(x + w/2, shuf, w, label="TIME-SHUFFLED", color="0.7", edgecolor="black", lw=0.5, hatch="xx")
ax[1,1].axhline(1.0, color=OKABE_ITO[5], ls="--", lw=1.5, path_effects=stroke())
for i, r in enumerate(REGIONS):
    ax[1,1].text(i, max(real[i], shuf[i])+0.01, f"gap {lam[r]['real_minus_shuffle']:+.3f}",
                 ha="center", fontsize=8, color="0.3")
ax[1,1].set(xticks=x, xticklabels=REGIONS, ylim=(0.85, 1.06), ylabel="|λ|max of the fitted flow",
            title="(d) Flow geometry is the SMOOTHER, not the dynamics\n(shuffling TIME barely changes |λ| → RETIRED)")
ax[1,1].legend(frameon=False, fontsize=8, loc="lower right")

fig.suptitle("rSLDS on session 5c0c560e — 40 ms, K = 3/4/3   |   3 CERTIFIED panels + 1 RETIRED",
             fontsize=12, y=1.005)
fig.tight_layout()
print("saved ->", savefig(fig, PHASE, "certified_results"))
plt.show()


In [ ]:
# ============================================================================
# FIGURE 2 — a LIVE fit: the regime sequence unfolding over the trial, per area
# (this is the static version of the "when does each area switch" question)
# ============================================================================
import time
fits = {}
t0 = time.time()
for r in REGIONS:
    K = K_PER_REGION[r]
    x, _, _ = counts_to_latents(D["regions"][r]["counts"], n_latent=3, smooth_sigma_ms=40.0, bin_size_s=0.04)
    fits[r] = (fit_rslds(x, K, recurrent=True, n_iter=60, n_restarts=3, seed=0, sticky=2.0), x, K)
print(f"live exact-EM fits done [{time.time()-t0:.0f}s]")

fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for a, r in zip(ax, REGIONS):
    res, _, K = fits[r]
    # occupancy P(regime | time): fraction of trials in each regime at each bin
    occ = np.stack([(res.states == k).mean(axis=0) for k in range(K)])   # (K, bins)
    cols = state_colors(K)
    a.stackplot(BC, occ, colors=cols, alpha=0.9, edgecolor="white", lw=0.3)
    a.axvline(0, color="0.15", lw=1.4, ls="--")
    # median entry time of each regime -> a marker row along the top
    for k in range(K):
        hit = res.states == k
        anyhit = hit.any(axis=1)
        if anyhit.sum() >= 5:
            ent = BC[hit.argmax(axis=1)[anyhit]]
            a.plot(np.median(ent), 1.06, "v", color=cols[k], markersize=9,
                   markeredgecolor="black", markeredgewidth=0.6, clip_on=False)
    a.set(ylabel=f"{r}\nP(regime | t)", ylim=(0, 1), xlim=(BC[0], BC[-1]))
    a.text(0.01, 0.9, f"K={K}", transform=a.transAxes, fontsize=9, color="0.3")
ax[0].set_title("Regime occupancy over the trial — the SEQUENCE made visible\n"
                "(▼ = median first-entry time of each regime; dashed = stimulus onset)")
ax[-1].set_xlabel("time from stimulus onset (s)")
fig.tight_layout()
print("saved ->", savefig(fig, PHASE, "regime_sequence_over_time"))
plt.show()

# cross-region ordering of median regime entry (the visual->motor cascade, direction-agnostic)
print("\nMedian first-entry time of each regime (s from stimulus onset):")
for r in REGIONS:
    res, _, K = fits[r]
    ents = []
    for k in range(K):
        hit = res.states == k; anyhit = hit.any(axis=1)
        ents.append(np.median(BC[hit.argmax(axis=1)[anyhit]]) if anyhit.sum()>=5 else np.nan)
    print(f"  {r:5s}: " + "  ".join(f"regime{k}={e:+.3f}" for k,e in enumerate(ents)))


In [ ]:
# ============================================================================
# FIGURE 3 — the latent trajectories, coloured by regime.
# ILLUSTRATIVE ONLY: per Fig 1(d), the flow GEOMETRY (attractor vs integrator) is a
# smoothing artefact here. Shown so you can see the regimes carve the latent, NOT to
# read attractor shapes off.
# ============================================================================
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, r in zip(ax, REGIONS):
    res, x, K = fits[r]
    cols = state_colors(K)
    # trial-averaged trajectory in PC1-PC2, with per-bin regime = modal regime across trials
    xbar = x.mean(axis=0)                          # (bins, D)
    modal = np.array([np.bincount(res.states[:, t], minlength=K).argmax() for t in range(x.shape[1])])
    a.plot(xbar[:, 0], xbar[:, 1], "-", color="0.6", lw=1, zorder=1)
    a.scatter(xbar[:, 0], xbar[:, 1], c=[cols[m] for m in modal], s=45, zorder=3,
              edgecolors="black", linewidths=0.5)
    a.scatter(xbar[0, 0], xbar[0, 1], marker="o", s=140, facecolor="none",
              edgecolor="black", lw=1.8, zorder=4, label="t = start")
    a.scatter(xbar[-1, 0], xbar[-1, 1], marker="s", s=120, facecolor="none",
              edgecolor="black", lw=1.8, zorder=4, label="t = end")
    lam = pd.DataFrame(screen["rows"]).query("alignment=='stimOn' and region==@r")["max_abs_lambda"].iloc[0]
    a.set(xlabel="latent PC1", ylabel="latent PC2",
          title=f"{r}  (K={K}, |λ|={lam:.2f} — geometry NOT interpretable)")
    a.legend(frameon=False, fontsize=8, loc="best")
fig.suptitle("Trial-averaged latent trajectory, coloured by modal regime — ILLUSTRATIVE "
             "(flow shape is a smoothing artefact; the SEQUENCE is the result)", y=1.02, fontsize=11)
fig.tight_layout()
print("saved ->", savefig(fig, PHASE, "latent_trajectories_illustrative"))
plt.show()
